# 05 Lead Scoring Model

This notebook is the fifth step of the **AI-Powered Lead Generation & Recommendation Analytics Platform** project.

In the previous notebook, I created an LLM-inspired intent classification dataset with structured fields such as:

- sentiment;
- intent category;
- purchase intent;
- lead score;
- high-intent lead flag.

In this notebook, I build a predictive lead scoring model to identify high-intent users based on user behaviour, review signals, pricing features, delivery experience, and intent classification results.

The goal is not only to predict high-intent leads, but also to understand which features are most important for lead quality and downstream recommendation strategy design.

This notebook covers:

1. Feature preparation;
2. Train/test split;
3. Logistic Regression baseline model;
4. Random Forest model;
5. Model evaluation;
6. Feature importance analysis;
7. Final lead score dataset export.

## Step 1: Import Libraries and Define Project Paths

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Set random seed for reproducibility
np.random.seed(42)

# Define project paths
BASE_DIR = Path("..")

FINAL_DIR = BASE_DIR / "data" / "final"
OUTPUT_DIR = BASE_DIR / "outputs"
TABLE_OUTPUT_DIR = OUTPUT_DIR / "tables"
MODEL_OUTPUT_DIR = OUTPUT_DIR / "model_results"

# Create folders if they do not exist
FINAL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Libraries imported and project paths defined successfully.")

Libraries imported and project paths defined successfully.


## Step 2: Load Intent Classification Dataset

In [3]:
# Load intent classification dataset
fact_reviews_llm = pd.read_csv(FINAL_DIR / "fact_reviews_llm.csv")

# Convert datetime column
fact_reviews_llm["order_time"] = pd.to_datetime(fact_reviews_llm["order_time"])

print("Dataset loaded successfully.")
print("Dataset shape:", fact_reviews_llm.shape)

fact_reviews_llm.head()

Dataset loaded successfully.
Dataset shape: (112650, 27)


,order_id,user_id,product_id,seller_id,category,order_time,price,price_band,gmv,review_score,...,viewed,clicked,added_to_cart,inquired,purchased,sentiment,intent_category,purchase_intent,lead_score,high_intent_flag
0,00010242fe8c5a6d1ba2dd792cb16214,871766c5855e863f6eccc05f988b23cb,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,cool_stuff,2017-09-13 08:59:02,58.90,Medium,72.19,5.0,...,1.0,1.0,1.0,1.0,1.0,positive,ready_to_purchase,high,100.0,1
1,00018f77f2f0320c557190d7a144bdd3,eb28e67c4c0b83846050ddfb8a35d051,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,pet_shop,2017-04-26 10:53:06,239.90,Premium,259.83,4.0,...,1.0,0.0,0.0,1.0,1.0,positive,price_sensitive,high,100.0,1
2,000229ec398224ef6ca0657da4fc703e,3818d81c6709e39d06b2738a8d3a2474,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,furniture_decor,2018-01-14 14:33:31,199.00,Premium,216.87,5.0,...,1.0,1.0,1.0,1.0,1.0,positive,delivery_concern,high,100.0,1
3,00024acbcdf0a6daa1e931b038114c75,af861d436cfc08b2c2ddefd0ba074622,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,perfumery,2018-08-08 10:00:35,12.99,Low,25.78,4.0,...,1.0,0.0,0.0,0.0,1.0,positive,ready_to_purchase,high,98.0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,64b576fb70d441e8f1b2d7d446e483c5,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,garden_tools,2017-02-04 13:57:51,199.90,Premium,218.04,5.0,...,1.0,1.0,1.0,1.0,1.0,positive,delivery_concern,high,100.0,1


## Step 3: Inspect Target Variable

The target variable is `high_intent_flag`.

Definition:

- `1`: high-intent lead;
- `0`: non-high-intent lead.

Before modelling, I inspect the class distribution to understand whether the dataset is balanced or imbalanced.

In [4]:
# Check target variable distribution
target_distribution = (
    fact_reviews_llm["high_intent_flag"]
    .value_counts()
    .reset_index()
)

target_distribution.columns = ["high_intent_flag", "records"]
target_distribution["percentage"] = (
    target_distribution["records"] / target_distribution["records"].sum()
).round(4)

target_distribution

,high_intent_flag,records,percentage
0,1,101106,0.8975
1,0,11544,0.1025


## Step 4: Select Modelling Features

I select features that are available before or during the lead generation process.

Feature groups:

### Behaviour features

- total events;
- viewed;
- clicked;
- added to cart;
- inquired;
- purchased.

### Product and transaction features

- price;
- GMV;
- review score;
- delivery delay days.

### Categorical features

- category;
- price band;
- traffic source;
- device type;
- user value segment;
- sentiment;
- intent category;
- purchase intent.

These features help the model learn how user behaviour and intent signals relate to high-intent lead quality.

In [5]:
# Define numerical and categorical features
numeric_features = [
    "price",
    "gmv",
    "review_score",
    "delivery_delay_days",
    "total_events",
    "viewed",
    "clicked",
    "added_to_cart",
    "inquired",
    "purchased"
]

categorical_features = [
    "category",
    "price_band",
    "traffic_source",
    "device_type",
    "user_value_segment",
    "sentiment",
    "intent_category",
    "purchase_intent"
]

target = "high_intent_flag"

# Create modelling dataset
model_data = fact_reviews_llm[
    numeric_features + categorical_features + [target]
].copy()

# Fill missing numerical values
for col in numeric_features:
    model_data[col] = model_data[col].fillna(model_data[col].median())

# Fill missing categorical values
for col in categorical_features:
    model_data[col] = model_data[col].fillna("Unknown").astype(str)

print("Modelling dataset shape:", model_data.shape)
model_data.head()

Modelling dataset shape: (112650, 19)


,price,gmv,review_score,delivery_delay_days,total_events,viewed,clicked,added_to_cart,inquired,purchased,category,price_band,traffic_source,device_type,user_value_segment,sentiment,intent_category,purchase_intent,high_intent_flag
0,58.90,72.19,5.0,-9.0,5.0,1.0,1.0,1.0,1.0,1.0,cool_stuff,Medium,Creator Profile,iOS,Low Value,positive,ready_to_purchase,high,1
1,239.90,259.83,4.0,-3.0,3.0,1.0,0.0,0.0,1.0,1.0,pet_shop,Premium,Search,Web,High Value,positive,price_sensitive,high,1
2,199.00,216.87,5.0,-14.0,5.0,1.0,1.0,1.0,1.0,1.0,furniture_decor,Premium,Search,Web,High Value,positive,delivery_concern,high,1
3,12.99,25.78,4.0,-6.0,2.0,1.0,0.0,0.0,0.0,1.0,perfumery,Low,Live Stream,iOS,Low Value,positive,ready_to_purchase,high,1
4,199.90,218.04,5.0,-16.0,5.0,1.0,1.0,1.0,1.0,1.0,garden_tools,Premium,Creator Profile,iOS,High Value,positive,delivery_concern,high,1


## Step 5: Train/Test Split

I split the dataset into training and testing sets.

- Training set: used to train the model;
- Testing set: used to evaluate model performance on unseen data.

I use stratified sampling so that the high-intent and non-high-intent classes are represented similarly in both sets.

In [7]:
# Define X and y
X = model_data[numeric_features + categorical_features]
y = model_data[target]

# Train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(4))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True).round(4))

Training set shape: (84487, 18)
Testing set shape: (28163, 18)

Training target distribution:
high_intent_flag
1    0.8975
0    0.1025
Name: proportion, dtype: float64

Testing target distribution:
high_intent_flag
1    0.8975
0    0.1025
Name: proportion, dtype: float64


## Step 6: Build Preprocessing Pipeline

Machine learning models require numerical input.

I use a preprocessing pipeline to:

1. Standardise numerical features;
2. One-hot encode categorical features.

This makes the modelling process clean, repeatable, and easier to maintain.

In [8]:
# Preprocessing for numerical and categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


## Step 7: Train Logistic Regression Baseline Model

Logistic Regression is used as a baseline model.

Why this model is useful:

1. It is simple and interpretable;
2. It provides a strong baseline for classification;
3. It is commonly used in business analytics for binary prediction tasks.

The model predicts whether a user-order record is a high-intent lead.

In [9]:
# Build Logistic Regression pipeline
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
    ]
)

# Train model
logistic_model.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


## Step 8: Train Random Forest Model

Random Forest is used as a more flexible model.

Why this model is useful:

1. It can capture non-linear relationships;
2. It handles interactions between features;
3. It provides feature importance scores;
4. It is suitable for understanding key drivers of high-intent leads.

This model will be used as the main lead scoring model.

In [10]:
# Build Random Forest pipeline
random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            max_depth=12,
            min_samples_split=20,
            min_samples_leaf=10,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ]
)

# Train model
random_forest_model.fit(X_train, y_train)

print("Random Forest model trained successfully.")

Random Forest model trained successfully.


## Step 9: Define Model Evaluation Function

I define a reusable function to evaluate model performance.

Metrics:

- Accuracy: overall correctness;
- Precision: how many predicted high-intent leads are actually high-intent;
- Recall: how many real high-intent leads are captured;
- F1 score: balance between precision and recall;
- ROC-AUC: ranking quality of predicted probabilities.

In a lead generation context, recall is important because missing high-intent users may result in lost conversion opportunities. Precision is also important because recommending to low-quality leads may waste exposure and seller resources.

In [12]:
def evaluate_model(model, X_test, y_test, model_name):
    """
    Evaluate a classification model and return a dictionary of metrics.
    """
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = {
        "model_name": model_name,
        "accuracy": round(accuracy_score(y_test, y_pred), 4),
        "precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "recall": round(recall_score(y_test, y_pred, zero_division=0), 4),
        "f1_score": round(f1_score(y_test, y_pred, zero_division=0), 4),
        "roc_auc": round(roc_auc_score(y_test, y_proba), 4)
    }
    
    return metrics

## Step 10: Evaluate Models

I evaluate both Logistic Regression and Random Forest on the test set.

The comparison helps determine whether the more flexible Random Forest model improves predictive performance over the baseline model.

In [13]:
# Evaluate both models
logistic_metrics = evaluate_model(
    logistic_model,
    X_test,
    y_test,
    "Logistic Regression"
)

rf_metrics = evaluate_model(
    random_forest_model,
    X_test,
    y_test,
    "Random Forest"
)

model_metrics = pd.DataFrame([logistic_metrics, rf_metrics])

model_metrics

,model_name,accuracy,precision,recall,f1_score,roc_auc
0,Logistic Regression,0.9997,1.0,0.9996,0.9998,1.0
1,Random Forest,0.9987,1.0,0.9985,0.9992,1.0


In [14]:
# Export model metrics
model_metrics.to_csv(
    MODEL_OUTPUT_DIR / "lead_scoring_model_metrics.csv",
    index=False
)

print("lead_scoring_model_metrics.csv exported successfully.")

lead_scoring_model_metrics.csv exported successfully.


## Step 11: Inspect Classification Report

I inspect the classification report for the Random Forest model.

This report provides precision, recall, and F1-score for each class.

Class interpretation:

- `0`: non-high-intent lead;
- `1`: high-intent lead.

In [15]:
# Generate classification report for Random Forest
rf_y_pred = random_forest_model.predict(X_test)

print(classification_report(y_test, rf_y_pred, zero_division=0))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99      2886
           1       1.00      1.00      1.00     25277

    accuracy                           1.00     28163
   macro avg       0.99      1.00      1.00     28163
weighted avg       1.00      1.00      1.00     28163



## Step 12: Confusion Matrix

The confusion matrix helps us understand prediction errors.

It shows:

- True negatives;
- False positives;
- False negatives;
- True positives.

In lead generation, false negatives are important because they represent high-intent users that the model failed to identify.

In [16]:
# Confusion matrix for Random Forest
cm = confusion_matrix(y_test, rf_y_pred)

confusion_matrix_df = pd.DataFrame(
    cm,
    index=["Actual Non-High Intent", "Actual High Intent"],
    columns=["Predicted Non-High Intent", "Predicted High Intent"]
)

confusion_matrix_df

,Predicted Non-High Intent,Predicted High Intent
Actual Non-High Intent,2886,0
Actual High Intent,38,25239


## Step 13: Extract Feature Names

To interpret feature importance, I first extract the transformed feature names from the preprocessing pipeline.

This includes:

- original numerical feature names;
- one-hot encoded categorical feature names.

In [17]:
# Get trained preprocessor from Random Forest pipeline
trained_preprocessor = random_forest_model.named_steps["preprocessor"]

# Get numerical feature names
num_feature_names = numeric_features

# Get one-hot encoded categorical feature names
cat_encoder = trained_preprocessor.named_transformers_["cat"]
cat_feature_names = cat_encoder.get_feature_names_out(categorical_features).tolist()

# Combine all transformed feature names
all_feature_names = num_feature_names + cat_feature_names

print("Total transformed features:", len(all_feature_names))
all_feature_names[:20]

Total transformed features: 114


['price',
 'gmv',
 'review_score',
 'delivery_delay_days',
 'total_events',
 'viewed',
 'clicked',
 'added_to_cart',
 'inquired',
 'purchased',
 'category_agro_industry_and_commerce',
 'category_air_conditioning',
 'category_art',
 'category_arts_and_craftmanship',
 'category_audio',
 'category_auto',
 'category_baby',
 'category_bed_bath_table',
 'category_books_general_interest',
 'category_books_imported']

## Step 14: Feature Importance Analysis

Random Forest provides feature importance scores.

This helps answer an important product question: 
**What signals are most useful for identifying high-intent leads?**

Potential important drivers may include:

- purchase intent;
- inquiry behaviour;
- add-to-cart behaviour;
- sentiment;
- review score;
- price band;
- traffic source;
- user value segment.

In [18]:
# Extract feature importance from Random Forest classifier
rf_classifier = random_forest_model.named_steps["classifier"]

feature_importance = pd.DataFrame({
    "feature": all_feature_names,
    "importance": rf_classifier.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="importance",
    ascending=False
).reset_index(drop=True)

feature_importance.head(20)

,feature,importance
0,purchase_intent_high,0.258483
1,purchase_intent_low,0.138921
2,review_score,0.104234
3,sentiment_positive,0.092779
4,added_to_cart,0.078552
5,total_events,0.060416
6,purchase_intent_medium,0.050234
7,intent_category_ready_to_purchase,0.049658
8,inquired,0.044061
9,sentiment_negative,0.033738


In [19]:
# Export feature importance
feature_importance.to_csv(
    MODEL_OUTPUT_DIR / "lead_scoring_feature_importance.csv",
    index=False
)

print("lead_scoring_feature_importance.csv exported successfully.")

lead_scoring_feature_importance.csv exported successfully.


## Step 15: Generate Predicted Lead Scores

The Random Forest model outputs a predicted probability that each record is a high-intent lead.

I convert this probability into a model-based lead score from 0 to 100.

This creates a model-driven scoring layer that can be compared with the earlier rule-based lead score.

In [20]:
# Predict high-intent probability for all records
lead_probability = random_forest_model.predict_proba(X)[:, 1]

# Convert probability to model lead score
fact_reviews_llm["model_high_intent_probability"] = lead_probability
fact_reviews_llm["model_lead_score"] = (lead_probability * 100).round(2)

# Create model-based high intent prediction
fact_reviews_llm["model_high_intent_prediction"] = np.where(
    fact_reviews_llm["model_high_intent_probability"] >= 0.5,
    1,
    0
)

fact_reviews_llm[[
    "user_id",
    "order_id",
    "lead_score",
    "model_high_intent_probability",
    "model_lead_score",
    "high_intent_flag",
    "model_high_intent_prediction"
]].head()

,user_id,order_id,lead_score,model_high_intent_probability,model_lead_score,high_intent_flag,model_high_intent_prediction
0,871766c5855e863f6eccc05f988b23cb,00010242fe8c5a6d1ba2dd792cb16214,100.0,1.000000,100.00,1,1
1,eb28e67c4c0b83846050ddfb8a35d051,00018f77f2f0320c557190d7a144bdd3,100.0,0.997691,99.77,1,1
2,3818d81c6709e39d06b2738a8d3a2474,000229ec398224ef6ca0657da4fc703e,100.0,0.999137,99.91,1,1
3,af861d436cfc08b2c2ddefd0ba074622,00024acbcdf0a6daa1e931b038114c75,98.0,0.989320,98.93,1,1
4,64b576fb70d441e8f1b2d7d446e483c5,00042b26cf59d7ce69dfabb4e55b4fd9,100.0,0.998073,99.81,1,1


## Step 16: Compare Rule-Based Lead Score and Model-Based Lead Score

The previous notebook created a transparent rule-based lead score.

This notebook creates a model-based lead score.

Comparing the two scores helps understand whether the machine learning model is aligned with the interpretable business logic.

In [21]:
# Compare rule-based and model-based lead scores
score_comparison = fact_reviews_llm[[
    "lead_score",
    "model_lead_score",
    "high_intent_flag"
]].describe()

score_correlation = fact_reviews_llm[[
    "lead_score",
    "model_lead_score"
]].corr().iloc[0, 1]

print("Correlation between rule-based lead score and model-based lead score:", round(score_correlation, 4))

score_comparison

Correlation between rule-based lead score and model-based lead score: 0.8567


,lead_score,model_lead_score,high_intent_flag
count,112650.000000,112650.000000,112650.000000
mean,89.973569,88.842098,0.897523
std,21.910071,29.817734,0.303276
min,0.000000,0.020000,0.000000
25%,98.000000,98.290000,1.000000
50%,100.000000,99.800000,1.000000
75%,100.000000,99.970000,1.000000
max,100.000000,100.000000,1.000000


## Step 17: Create Lead Score Summary by Segment

This table summarises model-based lead quality by user segment.

It helps evaluate whether different user groups have different lead quality patterns.

In [22]:
lead_score_model_summary = (
    fact_reviews_llm
    .groupby("user_value_segment")
    .agg(
        records=("order_id", "count"),
        unique_users=("user_id", "nunique"),
        avg_rule_based_lead_score=("lead_score", "mean"),
        avg_model_lead_score=("model_lead_score", "mean"),
        avg_high_intent_probability=("model_high_intent_probability", "mean"),
        actual_high_intent_rate=("high_intent_flag", "mean"),
        predicted_high_intent_rate=("model_high_intent_prediction", "mean"),
        total_gmv=("gmv", "sum")
    )
    .reset_index()
)

lead_score_model_summary["avg_rule_based_lead_score"] = lead_score_model_summary["avg_rule_based_lead_score"].round(2)
lead_score_model_summary["avg_model_lead_score"] = lead_score_model_summary["avg_model_lead_score"].round(2)
lead_score_model_summary["avg_high_intent_probability"] = lead_score_model_summary["avg_high_intent_probability"].round(4)
lead_score_model_summary["actual_high_intent_rate"] = lead_score_model_summary["actual_high_intent_rate"].round(4)
lead_score_model_summary["predicted_high_intent_rate"] = lead_score_model_summary["predicted_high_intent_rate"].round(4)
lead_score_model_summary["total_gmv"] = lead_score_model_summary["total_gmv"].round(2)

lead_score_model_summary

,user_value_segment,records,unique_users,avg_rule_based_lead_score,avg_model_lead_score,avg_high_intent_probability,actual_high_intent_rate,predicted_high_intent_rate,total_gmv
0,High Value,42766,31118,90.60,91.45,0.9145,0.9258,0.9255,10454977.90
1,Low Value,32229,31120,91.97,89.67,0.8967,0.9038,0.9023,1552691.00
2,Medium Value,35202,31120,91.52,90.27,0.9027,0.9099,0.9087,3412104.85
3,Unknown,2453,2180,30.50,11.96,0.1196,0.1447,0.1154,423779.49


In [23]:
# Export lead score model summary
lead_score_model_summary.to_csv(
    MODEL_OUTPUT_DIR / "lead_score_model_summary_by_segment.csv",
    index=False
)

print("lead_score_model_summary_by_segment.csv exported successfully.")

lead_score_model_summary_by_segment.csv exported successfully.


In [24]:
# Export final lead score dataset
fact_lead_scores = fact_reviews_llm.copy()

fact_lead_scores.to_csv(
    FINAL_DIR / "fact_lead_scores.csv",
    index=False
)

print("fact_lead_scores.csv exported successfully.")
print("File saved to:", FINAL_DIR / "fact_lead_scores.csv")

fact_lead_scores.csv exported successfully.
File saved to: ../data/final/fact_lead_scores.csv


## Model Interpretation Note

The model performance is extremely high because the target label `high_intent_flag` was created using interpretable business rules from the previous LLM-inspired intent classification stage. Some model features, such as `purchase_intent`, `sentiment`, and `intent_category`, are directly related to the label construction logic.

Therefore, the model should be interpreted as a validation of the lead scoring framework rather than as evidence of real-world causal prediction performance.

In a real production environment, the model should be trained on independently observed downstream outcomes, such as future inquiry, conversion, repeat purchase, or seller-side lead acceptance.

## Step 18: Summary of Lead Scoring Model

In this notebook, I built a predictive lead scoring model using user behaviour, product, review, delivery, and intent classification features.

Key outputs generated:

- `fact_lead_scores.csv`
- `lead_scoring_model_metrics.csv`
- `lead_scoring_feature_importance.csv`
- `lead_score_model_summary_by_segment.csv`

Business value of this stage:

1. Built a predictive model to identify high-intent leads;
2. Compared Logistic Regression and Random Forest models;
3. Evaluated the model using accuracy, precision, recall, F1-score, and ROC-AUC;
4. Identified key lead quality drivers through feature importance analysis;
5. Created a model-based lead score for downstream recommendation strategy design.

This stage connects user interaction behaviour modelling with recommendation strategy improvement, which is central to a lead generation product analytics workflow.